# Nemotron v7 — Training Notebook

**Trains a LoRA adapter and saves ALL adapter files as a zip.**

The output `adapter.zip` contains:
- `adapter_model.safetensors` — trained LoRA weights
- `adapter_config.json` — LoRA configuration (patched base_model_name_or_path)
- Any other files PEFT generates (README.md, etc.)

Use this with a separate submission notebook, or download the zip to iterate locally.

### v7 Config (proven 0.69 baseline + 0.86 improvements)
| Parameter | Value | Source |
|-----------|-------|--------|
| `lora_alpha` | 64 | 0.69 notebook (2:1 ratio) |
| `learning_rate` | 2e-5 | 0.69 notebook (eff LR = 4e-5) |
| `num_epochs` | 3 | 0.69 notebook |
| `targets` | q,k,v,o,in,out,up,down + **lm_head** | 0.69 + 0.86 |
| `dropout` | 0.0 | Both notebooks |

### Required Kaggle Inputs
- **Model**: `metric/nemotron-3-nano-30b-a3b-bf16`
- **Dataset**: `nemotron-cot-v5` (train_cot_v5_merged.jsonl)
- **Packages**: `nvidia-nemotron-offline-packages`
- **Accelerator**: GPU T4/P100/L4/A100

In [ ]:
# ============================================================
# 1. OFFLINE DEPENDENCY INSTALLATION
# ============================================================
import subprocess, sys, os
from pathlib import Path

def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read().strip()
            rel_pack_path = pth_file.parent / relpath
            if rel_pack_path.exists():
                sys.path.append(str(rel_pack_path))

offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir  = "/kaggle/working/packages"
os.makedirs(target_dir, exist_ok=True)

resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl", "peft"
    ])
    print("Offline packages installed.")

# wandb — must be set BEFORE import so it never tries to connect
os.environ["WANDB_MODE"] = "offline"

# Try installing wandb: offline packages first, then pip (if internet exists)
WANDB_AVAILABLE = False
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", offline_dir,
        "--target", target_dir,
        "wandb"
    ])
    WANDB_AVAILABLE = True
    print("wandb installed (offline).")
except Exception:
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "wandb"
        ])
        WANDB_AVAILABLE = True
        print("wandb installed (online).")
    except Exception:
        print("wandb not available — training will continue without W&B logging.")

sys.path.append(target_dir)
resolve_python_path(target_dir)

In [ ]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import stat, shutil, zipfile, time, json, re
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from tqdm.auto import tqdm

if WANDB_AVAILABLE:
    import wandb

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"W&B     : {'offline mode' if WANDB_AVAILABLE else 'disabled'}")

In [ ]:
# ============================================================
# 2b. WEIGHTS & BIASES — OFFLINE MODE (no internet needed)
# ============================================================
# W&B runs in offline mode: all metrics are saved locally to
# /kaggle/working/wandb/. After training, the run directory is
# zipped so you can download it and sync from your local machine:
#
#   wandb sync /path/to/wandb/offline-run-XXXXXXXX-XXXXXXXX
#
# This gives you full dashboard access (loss curves, LR schedule,
# grad norms, system metrics) — just delayed until you sync.

WANDB_PROJECT  = "nemotron-v7"
WANDB_RUN_NAME = "v7-lora-r32-a64-lr2e5-3ep"
WANDB_DIR      = "/kaggle/working"   # wandb creates ./wandb/ under this

if WANDB_AVAILABLE:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=WANDB_DIR,
        config={
            "model": "Nemotron-3-Nano-30B-A3B",
            "lora_rank": 32,
            "lora_alpha": 64,
            "learning_rate": 1e-4,
            "effective_lr": 4e-5,
            "num_epochs": 3,
            "batch_size": 1,
            "grad_accum": 8,
            "effective_batch": 16,
            "max_seq_len": 8192,
            "lora_targets": "q,k,v,o,in,out,up,down,lm_head",
            "lora_dropout": 0.0,
            "warmup_steps": 50,
            "scheduler": "cosine",
            "packing": True,
            "bf16": True,
        },
        tags=["nemotron", "lora", "v7", "sft"],
    )
    print(f"W&B offline run initialized: {wandb.run.dir}")
    print(f"After training, download wandb_logs.zip and run:")
    print(f"  wandb sync <run_dir>")
else:
    print("W&B not available — skipping init. Training metrics logged to stdout only.")

In [ ]:
# ============================================================
# 3. TRITON / RMSNORM FIXES (critical for NemotronH hybrid arch)
# ============================================================
# NemotronH uses Mamba-2 layers which rely on Triton kernels.
# The default rmsnorm_fn fails on some GPU configs — replace with pure PyTorch.

def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
# ============================================================
# 4. HYPERPARAMETERS — v7.2 (DoRA + rsLoRA + PiSSA + LoRA+)
# ============================================================
# Eval-server contract (vLLM):
#   max_lora_rank          = 32        -> r <= 32
#   max_tokens             = 7680
#   max_model_len          = 8192
#   temperature            = 0.0       -> greedy (dropout must be 0)
#   top_p                  = 1.0
#   max_num_seqs           = 64
#   gpu_memory_utilization = 0.85
#
# LR note: rsLoRA changes scaling from alpha/r -> alpha/sqrt(r), which
# inflates the effective LR ~5x at r=32. So we drop LR from 1e-4 -> 2e-5.

LORA_RANK           = 32          # eval-server cap
LORA_ALPHA          = 64          # 2:1 ratio
MAX_SEQ_LEN         = 7680        # eval-server max_tokens
NUM_EPOCHS          = 3           # LoRA sweet spot
BATCH_SIZE          = 4
GRAD_ACCUM          = 8           # effective batch = 32
LR                  = 2e-5        # lowered for rsLoRA (alpha/sqrt(r) scaling)
WARMUP_STEPS        = 50
SAVE_EVERY_N_EPOCHS = 1

# --- Adapter upgrades (all eval-safe; merge to standard LoRA) ---
USE_DORA       = True             # +1-3% : magnitude/direction decomposition
USE_RSLORA     = True             # +0.5-1.5% : stable scaling at r=32
PISSA_INIT     = "pissa_niter_4"  # +1-2% : SVD-aligned warm start
LORAPLUS_RATIO = 16               # +0.5-1%  : B trains 16x faster than A

# Stratified batching — one domain per batch
USE_STRATIFIED_BATCHING = True

MODEL_PATH  = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR  = "/kaggle/working/adapter"
CKPT_DIR    = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

OUR_DATA_PATHS = [
    "/kaggle/input/nemotron-cot-v5/train_cot_v5_merged.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v5_merged.jsonl",
    "/kaggle/input/nemotron-cot-v4/train_cot_v4_real.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v4_real.jsonl",
    "/kaggle/input/nemotron-dataset/train_cot_v3_metric_aligned.jsonl",
    "/kaggle/input/nemotron-cot-v3/train_cot_v3_metric_aligned.jsonl",
]
EXTERNAL_CSV_PATHS = [
    "/kaggle/input/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv",
    "/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv",
]

# Eval-server assertions
assert LORA_RANK   <= 32,   f"LORA_RANK={LORA_RANK} exceeds eval cap 32"
assert MAX_SEQ_LEN <= 7680, f"MAX_SEQ_LEN={MAX_SEQ_LEN} exceeds eval cap 7680"
if USE_RSLORA:
    assert LR <= 5e-5, f"LR={LR} too high with rsLoRA (alpha/sqrt(r) inflates effective LR)"

eff_lr = LR * LORA_ALPHA / LORA_RANK
print(f"Epochs     : {NUM_EPOCHS}  (checkpoint every {SAVE_EVERY_N_EPOCHS})")
print(f"LR         : {LR:.1e}  (effective: {eff_lr:.1e})")
print(f"Batch      : {BATCH_SIZE}×{GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} eff  |  rank={LORA_RANK}, alpha={LORA_ALPHA}")
print(f"Max seqlen : {MAX_SEQ_LEN}")
print(f"Warmup     : {WARMUP_STEPS} steps")
print(f"Stratified : {USE_STRATIFIED_BATCHING}")
print(f"Adapter    : DoRA={USE_DORA}, rsLoRA={USE_RSLORA}, init={PISSA_INIT}, LoRA+ ratio={LORAPLUS_RATIO}")
print(f"Ckpt dir   : {CKPT_DIR}")


In [ ]:
# ============================================================
# 5. CALLBACKS — progress bar + per-epoch checkpoint zip
# ============================================================

class LiveProgressCallback(TrainerCallback):
    """Live tqdm bar with loss + ETA."""
    def __init__(self):
        self.pbar       = None
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.pbar = tqdm(total=state.max_steps, desc="Training",
                         unit="step", dynamic_ncols=True, file=sys.stdout)
        self.start_time = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is None:
            return
        elapsed = time.time() - self.start_time
        step    = state.global_step
        eta     = (elapsed / step) * (state.max_steps - step) if step > 0 else 0
        loss_str = (f"loss={state.log_history[-1]['loss']:.4f}"
                    if state.log_history and "loss" in state.log_history[-1] else "loss=...")
        self.pbar.set_postfix_str(f"{loss_str}  elapsed={elapsed/60:.1f}m  eta={eta/60:.1f}m")
        self.pbar.update(1)
        sys.stdout.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar:
            self.pbar.close()


class CheckpointZipCallback(TrainerCallback):
    """After every N epochs: save adapter → patch config → zip to CKPT_DIR."""

    def __init__(self, ckpt_dir, output_dir, every_n=1):
        self.ckpt_dir   = ckpt_dir
        self.output_dir = output_dir
        self.every_n    = every_n
        self.epoch_losses = {}   # epoch → loss (for summary)

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = round(state.epoch)   # state.epoch is a float like 1.0, 2.0 …
        if epoch % self.every_n != 0:
            return

        # --- 1. Save adapter weights to a temp subfolder ---
        epoch_dir = os.path.join(self.output_dir, f"epoch_{epoch:02d}")
        os.makedirs(epoch_dir, exist_ok=True)
        save_kwargs = {}
            if 'PISSA_INIT' in globals() and PISSA_INIT and PISSA_INIT.startswith('pissa'):
                save_kwargs['path_initial_model_for_weight_conversion'] = MODEL_PATH
            model.save_pretrained(epoch_dir, **save_kwargs)

        # --- 2. Patch base_model_name_or_path ---
        cfg_path = os.path.join(epoch_dir, "adapter_config.json")
        with open(cfg_path) as f:
            cfg = json.load(f)
        cfg["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"
        with open(cfg_path, "w") as f:
            json.dump(cfg, f, indent=2)

        # --- 3. Zip ALL files in epoch_dir ---
        zip_name = f"adapter_epoch_{epoch:02d}.zip"
        zip_path = os.path.join(self.ckpt_dir, zip_name)
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in sorted(os.listdir(epoch_dir)):
                fp = os.path.join(epoch_dir, fname)
                if os.path.isfile(fp):
                    zf.write(fp, arcname=fname)

        # --- 4. Record loss and report ---
        recent_losses = [h["loss"] for h in state.log_history if "loss" in h]
        avg_loss = sum(recent_losses[-10:]) / len(recent_losses[-10:]) if recent_losses else float("nan")
        self.epoch_losses[epoch] = avg_loss

        zip_mb = os.path.getsize(zip_path) / 1024 / 1024
        print(f"\n[Epoch {epoch:02d}] ✓ {zip_name}  ({zip_mb:.1f} MB)  avg_loss={avg_loss:.4f}")

        # W&B log if available
        if WANDB_AVAILABLE and wandb.run is not None:
            wandb.log({"epoch_checkpoint/epoch": epoch,
                       "epoch_checkpoint/avg_loss": avg_loss,
                       "epoch_checkpoint/zip_mb": zip_mb}, step=state.global_step)

    def print_summary(self):
        if not self.epoch_losses:
            return
        print("\n  Checkpoint loss summary:")
        best_epoch = min(self.epoch_losses, key=self.epoch_losses.get)
        for ep, loss in sorted(self.epoch_losses.items()):
            marker = " ← best" if ep == best_epoch else ""
            print(f"    epoch {ep:02d}: loss={loss:.4f}{marker}")
        print(f"\n  Best checkpoint: adapter_epoch_{best_epoch:02d}.zip  (use this for submission)")


# instantiate — trainer gets this in cell 11
ckpt_callback = CheckpointZipCallback(
    ckpt_dir=CKPT_DIR, output_dir=OUTPUT_DIR, every_n=SAVE_EVERY_N_EPOCHS
)
print("Callbacks ready: LiveProgressCallback + CheckpointZipCallback")

In [ ]:
# ============================================================
# 6. LOAD DATASET — combine BOTH sources for max coverage
# ============================================================
print("Loading datasets...\n")

our_data = []
ext_csv_path = None

# 1) Try our metric-aligned JSONL (prioritized list)
for path in OUR_DATA_PATHS:
    if os.path.exists(path):
        print(f"Found our JSONL: {path}")
        with open(path, 'r') as f:
            for line in f:
                our_data.append(json.loads(line))
        print(f"  -> {len(our_data)} metric-aligned examples")
        break

if not our_data:
    print("Our JSONL not found (searched all candidate paths)")

# 2) Try external CSV as supplemental data
for path in EXTERNAL_CSV_PATHS:
    if os.path.exists(path):
        ext_csv_path = path
        print(f"Found external CSV: {path}")
        break

if not ext_csv_path:
    print("External CSV not found (optional)")

if not our_data and not ext_csv_path:
    raise FileNotFoundError(
        "No training data found!\n"
        "Upload train_cot_v5_merged.jsonl as Kaggle dataset named 'nemotron-cot-v5'\n"
        "OR add kienngx/nemotron-30b-competition-trainingdata-cot-labels as input"
    )

print(f"\nData sources: our_jsonl={len(our_data)}, ext_csv={'YES' if ext_csv_path else 'NO'}")

In [ ]:
# ============================================================
# 7. TOKENIZER & FORMAT — unified pipeline + category labels
# ============================================================
# We infer a category label per sample so the Cell 11 stratified sampler
# can keep each batch domain-pure (bit-ops in one batch, cipher in another, etc.)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

# --- Category inference ------------------------------------------------
# 6 known categories + fallback. Matches the keywords actually present in
# the puzzle prompts ("Alice's Wonderland" framing).
CAT_KEYWORDS = [
    ("bit_manipulation", ["binary", "bit", "8-bit", "byte", "XOR", "rotate"]),
    ("cipher",           ["cipher", "encrypt", "decrypt", "substitut"]),
    ("numeral",          ["roman", "numeral", "XVIII", "XXIV"]),
    ("unit_conversion",  ["convert", "unit", "meter", "mile", "kilogram", "liter"]),
    ("gravity",          ["gravity", "gravitational", "d = 0.5", "t²", "t^2", "seconds"]),
    ("transformation",   ["rule", "@", "&", "$", "transform", "operator"]),
]

def infer_category(prompt: str) -> str:
    p = prompt.lower()
    scores = {}
    for name, kws in CAT_KEYWORDS:
        s = sum(1 for k in kws if k.lower() in p)
        if s: scores[name] = s
    return max(scores, key=scores.get) if scores else "other"

all_texts  = []
all_labels = []

# --- Source A: Our metric-aligned JSONL ---
if our_data:
    print(f"Formatting {len(our_data)} JSONL examples...")
    for example in our_data:
        messages = [m for m in example['messages'] if m['role'] != 'system']
        user_content = messages[0]['content'] if messages and messages[0]['role'] == 'user' else ''

        # Ensure <think> tags wrap reasoning
        assistant_msg = messages[-1]
        if '<think>' not in assistant_msg['content']:
            content = assistant_msg['content']
            boxed_match = re.search(r'(\\boxed\{.*?\})\s*$', content)
            if boxed_match:
                reasoning = content[:boxed_match.start()].strip()
                boxed_answer = boxed_match.group(1)
                messages[-1] = {
                    'role': 'assistant',
                    'content': f"<think>\n{reasoning}\n</think>\n{boxed_answer}"
                }

        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            text = (
                f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n{messages[-1]['content']}<|im_end|>"
            )
        all_texts.append(text)
        # Prefer explicit label field if present on the example
        lbl = example.get('category') or example.get('label') or infer_category(user_content)
        all_labels.append(lbl)
    print(f"  -> {len(all_texts)} examples from JSONL")

# --- Source B: External CSV (deduplicated) ---
if ext_csv_path:
    import pandas as pd
    ext_df = pd.read_csv(ext_csv_path)
    print(f"\nExternal CSV: {len(ext_df)} rows, columns: {list(ext_df.columns)}")

    existing_prompts = set()
    if our_data:
        for example in our_data:
            user_content = example['messages'][0]['content'] if example['messages'][0]['role'] == 'user' else ''
            existing_prompts.add(user_content[:200])

    ext_added = 0
    for _, row in ext_df.iterrows():
        prompt = str(row.get('prompt', ''))
        if prompt[:200] in existing_prompts:
            continue

        answer = str(row.get('answer', ''))
        cot = str(row.get('generated_cot', ''))

        if not prompt.strip() or not answer.strip():
            continue

        user_msg = prompt + EVAL_SUFFIX
        assistant_msg = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"

        try:
            messages = [
                {'role': 'user',      'content': user_msg},
                {'role': 'assistant', 'content': assistant_msg},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            text = (
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
            )
        all_texts.append(text)
        # CSV sometimes has 'category' / 'type' column
        lbl = str(row.get('category', '') or row.get('type', '') or '').strip().lower()
        if not lbl or lbl == 'nan':
            lbl = infer_category(prompt)
        all_labels.append(lbl)
        ext_added += 1

    print(f"  -> {ext_added} NEW examples from external CSV (after dedup)")

# --- Build final dataset ---
hf_dataset = Dataset.from_dict({'text': all_texts, 'label': all_labels})
print(f"\nTOTAL DATASET: {len(hf_dataset)} examples")

# Label distribution
from collections import Counter
dist = Counter(all_labels)
print("\nCategory distribution:")
for name, n in dist.most_common():
    print(f"  {name:20s} {n:5d}  ({100*n/len(all_labels):.1f}%)")

print(f"\nSample (first 400 chars):")
print(hf_dataset[0]['text'][:400])


In [ ]:
# ============================================================
# 8. DROP OVERSIZED SAMPLES (> MAX_SEQ_LEN tokens)
# ============================================================
# MAX_SEQ_LEN = 8192 now → we keep nearly all CoT samples (vs ~dropping
# ~10% at 2048). The `label` column is preserved for stratified batching.
print(f"Filtering samples > {MAX_SEQ_LEN} tokens...")
before = len(hf_dataset)

def get_token_length(example):
    ids = tokenizer(example['text'], truncation=False,
                    return_attention_mask=False)['input_ids']
    return {'token_len': len(ids)}

hf_dataset = hf_dataset.map(get_token_length, desc="Counting tokens")
hf_dataset = hf_dataset.filter(
    lambda x: x['token_len'] <= MAX_SEQ_LEN,
    desc="Dropping oversized",
)
hf_dataset = hf_dataset.remove_columns(['token_len'])
print(f"  Kept {len(hf_dataset)} / {before}  ({before - len(hf_dataset)} dropped)")

steps_estimate = len(hf_dataset) // (BATCH_SIZE * GRAD_ACCUM) * NUM_EPOCHS
print(f"\nEstimated steps : {steps_estimate}")
print(f"Estimated time  : ~{steps_estimate * 8 / 3600:.1f} hrs")


In [ ]:
# ============================================================
# 9. LOAD MODEL — bf16, NO quantization
# ============================================================
# NemotronH = HYBRID architecture (52 layers):
#   - 23 Attention layers (Transformer)
#   - 23 Mamba-2 layers (SSM)
#   - 6 MoE layers (Mixture of Experts)
# Does NOT support flash_attention_2 — must use "eager" or default

flash_whl = "/kaggle/input/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
if os.path.exists(flash_whl):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-index", flash_whl])
        print("Installed flash_attn wheel (used by internal kernels)")
    except Exception as e:
        print(f"flash_attn install skipped: {e}")

print("Loading base model (bf16, eager attention)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

# Disable fast path for NemotronH
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False

print(f"Model loaded on GPU")

In [ ]:
# ============================================================
# 10. APPLY LoRA — v7.2: DoRA + rsLoRA + PiSSA warm-start
# ============================================================
# All three upgrades merge to a standard LoRA adapter at save time,
# so vLLM sees a normal rank-32 LoRA (eval-server compatible).
#
#   DoRA  (use_dora=True)          : learns per-neuron magnitude separately
#                                    from direction.  +1-3%.
#   rsLoRA (use_rslora=True)       : scales by alpha/sqrt(r) instead of
#                                    alpha/r -> stable grads at r=32.
#   PiSSA  (init_lora_weights=...) : inits A,B from top-r SVD of W so
#                                    training starts aligned with the
#                                    most important weight directions.
#                                    Needs special save (see Cell 13).
#
# modules_to_save=["embed_tokens"] keeps input embeddings in sync with
# the LoRA'd lm_head (tied-embedding safety).

LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention  (23 layers)
    "in_proj", "out_proj",                      # Mamba-2 SSM (23 layers)
    "up_proj", "down_proj",                     # MLP / MoE experts
    "lm_head",                                  # Output head
]

lora_config = LoraConfig(
    r=LORA_RANK,                         # eval cap: <= 32
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    modules_to_save=["embed_tokens"],
    lora_dropout=0.0,                    # greedy decoding => must be 0
    bias="none",
    task_type=TaskType.CAUSAL_LM,

    # ---- v7.2 upgrades ----
    use_dora=USE_DORA,                   # DoRA
    use_rslora=USE_RSLORA,               # rsLoRA
    init_lora_weights=PISSA_INIT,        # PiSSA SVD init (e.g. "pissa_niter_4")
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Triton compiler fix
try:
    import triton.backends.nvidia.compiler as nv_compiler
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print(f"Triton compiler fix skipped: {e}")


In [ ]:
# ============================================================
# 11. TRAINING — Stratified SFT with per-epoch checkpointing
# ============================================================
# Problem: a random batch that mixes bit-ops + cipher + gravity sends
# contradictory gradient signals on a tiny effective batch. We instead
# build a deterministic "one-domain-per-batch" index order.
#
# Algorithm (build_stratified_index_order):
#   1. Bucket sample indices by label.
#   2. Shuffle within each bucket (seeded for reproducibility).
#   3. Emit contiguous "chunks" of size = effective_batch from one bucket
#      at a time; round-robin across buckets so every domain is seen
#      frequently without letting any batch mix domains.
#   4. Repeat for each epoch with a different seed → different order
#      per epoch (data diversity preserved).

from torch.utils.data import Sampler
import random as _random

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM


def build_stratified_index_order(labels, chunk_size, seed=0):
    """Return an index list where each `chunk_size`-block is one label."""
    buckets = {}
    for i, lbl in enumerate(labels):
        buckets.setdefault(lbl, []).append(i)

    rng = _random.Random(seed)
    for lbl in buckets:
        rng.shuffle(buckets[lbl])

    # Round-robin: pop `chunk_size` indices from each non-empty bucket.
    order = []
    active = list(buckets.keys())
    rng.shuffle(active)
    while active:
        next_active = []
        for lbl in active:
            take = buckets[lbl][:chunk_size]
            buckets[lbl] = buckets[lbl][chunk_size:]
            order.extend(take)
            if buckets[lbl]:
                next_active.append(lbl)
        active = next_active
    return order


class PrecomputedOrderSampler(Sampler):
    """Emits a pre-built index order; re-shuffles (stratified) per epoch."""
    def __init__(self, labels, chunk_size, num_epochs, base_seed=1337):
        self.labels      = list(labels)
        self.chunk_size  = chunk_size
        self.num_epochs  = num_epochs
        self.base_seed   = base_seed
        self.epoch       = 0
        self._current_order = build_stratified_index_order(
            self.labels, chunk_size, seed=base_seed
        )

    def set_epoch(self, epoch):
        self.epoch = epoch
        self._current_order = build_stratified_index_order(
            self.labels, self.chunk_size, seed=self.base_seed + epoch
        )

    def __iter__(self):
        return iter(self._current_order)

    def __len__(self):
        return len(self.labels)


class StratifiedSFTTrainer(SFTTrainer):
    """SFTTrainer that uses a stratified sampler instead of random shuffle."""
    def __init__(self, *args, stratified_labels=None, chunk_size=16,
                 num_epochs=3, **kwargs):
        self._strat_labels = stratified_labels
        self._strat_chunk  = chunk_size
        self._strat_epochs = num_epochs
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, *args, **kwargs):
        if self._strat_labels is None:
            return super()._get_train_sampler(*args, **kwargs)
        return PrecomputedOrderSampler(
            labels=self._strat_labels,
            chunk_size=self._strat_chunk,
            num_epochs=self._strat_epochs,
        )


# -------- SFTConfig ---------------------------------------------------
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=10,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    save_strategy="no",               # CheckpointZipCallback handles saving
    report_to="wandb" if WANDB_AVAILABLE else "none",
    run_name=WANDB_RUN_NAME if WANDB_AVAILABLE else None,
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,                    # packing breaks per-sample label alignment
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    remove_unused_columns=False,      # keep 'label' column on dataset
    loraplus_lr_ratio=LORAPLUS_RATIO, # LoRA+ : lr_B = ratio * lr_A
)

# -------- Choose trainer ----------------------------------------------
labels_for_sampler = list(hf_dataset['label']) if USE_STRATIFIED_BATCHING else None

if USE_STRATIFIED_BATCHING:
    print(f"Using STRATIFIED batching (chunk = {EFFECTIVE_BATCH} samples/domain)")
    trainer = StratifiedSFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[LiveProgressCallback(), ckpt_callback],
        stratified_labels=labels_for_sampler,
        chunk_size=EFFECTIVE_BATCH,
        num_epochs=NUM_EPOCHS,
    )
else:
    print("Using standard RANDOM batching")
    trainer = SFTTrainer(
        model=model,
        train_dataset=hf_dataset,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[LiveProgressCallback(), ckpt_callback],
    )

steps_per_epoch = trainer.state.max_steps // NUM_EPOCHS if hasattr(trainer.state, 'max_steps') else "?"
print(f"\nStarting training:")
print(f"  {len(hf_dataset)} samples  |  {NUM_EPOCHS} epochs  |  max_seq_len={MAX_SEQ_LEN}")
print(f"  LR={LR:.1e} (eff={LR*LORA_ALPHA/LORA_RANK:.1e})  warmup={WARMUP_STEPS} steps")
print(f"  batch={BATCH_SIZE}×{GRAD_ACCUM}={EFFECTIVE_BATCH} eff  |  steps/epoch≈{steps_per_epoch}")
print(f"  Stratified: {USE_STRATIFIED_BATCHING}  |  Checkpoint every {SAVE_EVERY_N_EPOCHS} epoch(s) → {CKPT_DIR}")
print(f"  W&B: {'offline logging' if WANDB_AVAILABLE else 'disabled'}\n")

t0 = time.time()
trainer.train()
elapsed_hrs = (time.time() - t0) / 3600
print(f"\nTraining complete! Time: {elapsed_hrs:.2f} hrs")

ckpt_callback.print_summary()


In [ ]:
# ============================================================
# 12. SAVE ADAPTER — PiSSA-aware conversion to standard LoRA
# ============================================================
# PiSSA init modifies the residual  W <- W - B*A_init  during training,
# so A,B alone are NOT a valid standalone adapter until we "convert"
# them back against the original base model. PEFT does this when we
# pass `path_initial_model_for_weight_conversion=MODEL_PATH`.
#
# After this call, adapter_model.safetensors is a plain rank-32 LoRA
# that vLLM can load without any PiSSA awareness.

save_kwargs = {}
if PISSA_INIT and PISSA_INIT.startswith("pissa"):
    save_kwargs["path_initial_model_for_weight_conversion"] = MODEL_PATH
    print(f"Saving with PiSSA -> standard-LoRA conversion (base: {MODEL_PATH})")

trainer.model.save_pretrained(OUTPUT_DIR, **save_kwargs)

# Fix base_model_name_or_path to canonical Kaggle model name
config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
with open(config_path) as f:
    adapter_config = json.load(f)

adapter_config["base_model_name_or_path"] = "metric/nemotron-3-nano-30b-a3b-bf16"

with open(config_path, "w") as f:
    json.dump(adapter_config, f, indent=2)

print(f"base_model_name_or_path -> {adapter_config['base_model_name_or_path']}")
print(f"peft_type              -> {adapter_config.get('peft_type')}")
print(f"r / alpha              -> {adapter_config.get('r')} / {adapter_config.get('lora_alpha')}")
print(f"use_dora               -> {adapter_config.get('use_dora')}")
print(f"use_rslora             -> {adapter_config.get('use_rslora')}")

# Verify weights look trained (non-zero norms)
try:
    from safetensors import safe_open
    with safe_open(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"),
                   framework="pt") as f:
        keys  = list(f.keys())
        norms = [f.get_tensor(k).norm().item() for k in keys[:5]]
    print(f"\nAdapter tensors: {len(keys)} parameters")
    print(f"First 5 weight norms: {[f'{n:.4f}' for n in norms]}")
    if all(n < 0.001 for n in norms):
        print("WARNING: Norms near 0 — adapter may be untrained!")
    else:
        print("Adapter looks healthy (non-zero weights).")
except Exception as e:
    print(f"Could not verify safetensors: {e}")

# Show all files in adapter directory
print(f"\nFiles in {OUTPUT_DIR}:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f"  {fname}  ({size_mb:.2f} MB)")


In [ ]:
# ============================================================
# 13. ZIP FINAL ADAPTER + LIST ALL CHECKPOINTS
# ============================================================
# The FINAL adapter (after all epochs) is also zipped as adapter.zip.
# Per-epoch checkpoint zips are already in CKPT_DIR.
# Download the epoch zip with the lowest loss for best results.

ZIP_PATH = "/kaggle/working/adapter.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            file_count += 1

with zipfile.ZipFile(ZIP_PATH) as zf:
    contents = zf.namelist()

zip_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024

print(f"{'='*60}")
print(f"  FINAL adapter.zip  ({zip_mb:.1f} MB)  — {file_count} files")
print(f"  Contents: {contents}")
print(f"{'='*60}")

# List all per-epoch checkpoints
ckpt_zips = sorted(
    [f for f in os.listdir(CKPT_DIR) if f.endswith(".zip")],
    key=lambda x: int(x.replace("adapter_epoch_", "").replace(".zip", ""))
    if x.replace("adapter_epoch_", "").replace(".zip", "").isdigit() else 0
)

print(f"\nPer-epoch checkpoints in {CKPT_DIR}:")
total_ckpt_mb = 0
for zname in ckpt_zips:
    zpath = os.path.join(CKPT_DIR, zname)
    mb = os.path.getsize(zpath) / 1024 / 1024
    total_ckpt_mb += mb
    epoch_num = zname.replace("adapter_epoch_", "").replace(".zip", "")
    loss = ckpt_callback.epoch_losses.get(int(epoch_num), float("nan")) if epoch_num.isdigit() else float("nan")
    print(f"  {zname:<30}  {mb:6.1f} MB  loss={loss:.4f}")

print(f"\nTotal checkpoint storage: {total_ckpt_mb:.1f} MB  ({len(ckpt_zips)} files)")
print(f"\nTip: Use the checkpoint with the LOWEST loss for your submission.")

assert "adapter_config.json" in contents, "MISSING adapter_config.json!"
assert "adapter_model.safetensors" in contents, "MISSING adapter_model.safetensors!"

In [ ]:
# ============================================================
# 14. FINAL VERIFICATION — print config for sanity check
# ============================================================
print("=" * 60)
print("  v7 TRAINING SUMMARY")
print("=" * 60)

with open(os.path.join(OUTPUT_DIR, "adapter_config.json")) as f:
    final_cfg = json.load(f)

print(f"\n  Model             : Nemotron-3-Nano-30B-A3B (Hybrid)")
print(f"  base_model_name   : {final_cfg.get('base_model_name_or_path')}")
print(f"  LoRA rank (r)     : {final_cfg.get('r')}")
print(f"  LoRA alpha        : {final_cfg.get('lora_alpha')}")
print(f"  Alpha:Rank ratio  : {final_cfg.get('lora_alpha', 0)}:{final_cfg.get('r', 1)}")
print(f"  Target modules    : {final_cfg.get('target_modules')}")
print(f"  Dropout           : {final_cfg.get('lora_dropout')}")
print(f"  Task type         : {final_cfg.get('task_type')}")
print(f"\n  Dataset           : {len(hf_dataset)} examples (after filtering)")
print(f"  Epochs            : {NUM_EPOCHS}")
print(f"  Learning rate     : {LR}")
print(f"  Effective LR      : {LR * LORA_ALPHA / LORA_RANK}")
print(f"  Effective batch   : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Training time     : {elapsed_hrs:.2f} hrs")
print(f"\n  Adapter zip       : {ZIP_PATH} ({zip_mb:.1f} MB)")
print(f"  Adapter files     : {contents}")

# Verify critical settings
targets = final_cfg.get('target_modules', [])
checks = [
    ("base_model = metric/...",    final_cfg.get('base_model_name_or_path') == 'metric/nemotron-3-nano-30b-a3b-bf16'),
    ("out_proj IN targets",        'out_proj' in targets),
    ("lm_head IN targets",         'lm_head' in targets),
    ("gate_proj NOT in targets",   'gate_proj' not in targets),
    ("dropout = 0",                final_cfg.get('lora_dropout', -1) == 0.0),
    ("rank = 32",                  final_cfg.get('r') == 32),
    ("alpha = 64",                 final_cfg.get('lora_alpha') == 64),
]

print(f"\n  Verification checks:")
all_ok = True
for name, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_ok = False
    print(f"    [{status}] {name}")

if all_ok:
    print(f"\n  All checks passed!")
    print(f"  -> Download adapter.zip from Kaggle Output")
    print(f"  -> Use with a separate submission notebook, or submit directly")
else:
    print(f"\n  WARNING: Some checks failed. Review before using.")

# ---- W&B: log final metrics, finish run, zip logs for download ----
if WANDB_AVAILABLE:
    wandb.log({
        "final/training_hours": elapsed_hrs,
        "final/dataset_size": len(hf_dataset),
        "final/adapter_zip_mb": zip_mb,
        "final/adapter_file_count": len(contents),
    })
    wandb.finish()

    # Zip the entire wandb directory so you can download & sync locally
    wandb_dir = os.path.join(WANDB_DIR, "wandb")
    wandb_zip = "/kaggle/working/wandb_logs.zip"
    if os.path.exists(wandb_dir):
        with zipfile.ZipFile(wandb_zip, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(wandb_dir):
                for fname in files:
                    fpath = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, WANDB_DIR)
                    zf.write(fpath, arcname=arcname)
        wandb_zip_mb = os.path.getsize(wandb_zip) / 1024 / 1024
        print(f"\n  W&B logs zipped: {wandb_zip} ({wandb_zip_mb:.1f} MB)")
        print(f"  To view in dashboard:")
        print(f"    1. Download wandb_logs.zip from Kaggle Output")
        print(f"    2. Unzip it")
        print(f"    3. Run: wandb sync wandb/offline-run-*")
    else:
        print("\n  W&B directory not found — no logs to zip.")
else:
    print("\n  W&B was disabled — no logs to sync.")